## **Deploy Key**

Es especialmente útil cuando tienes un servidor y quieres que ese servidor pueda acceder a un repositorio de GitHub sin utilizar tu cuenta personal.

### **1. ¿Qué es una Deploy Key?**

Una Deploy Key es básicamente un par de claves SSH creado para que un servidor tenga acceso a un repositorio específico de GitHub.

```bash
                 GitHub
        ┌───────────────────────┐
        │  Mi repositorio       │
        │  proyecto.git         │
        │                       │
        │  Deploy Key pública   │
        └──────────┬────────────┘
                   │
                   │ SSH
                   │
                   ▼
        ┌──────────────────────┐
        │       SERVIDOR       │
        │                      │
        │ deploy_key           │ ← privada
        │ deploy_key.pub       │ ← pública
        └──────────────────────┘
```

La clave pública se registra en GitHub.

La clave privada permanece exclusivamente en el servidor. GitHub documenta precisamente este modelo: la clave pública queda asociada al repositorio y la privada permanece en el servidor

### **2. ¿Por qué no usar simplemente tu SSH normal?**

Supongamos que tú tienes:

```bash
PC personal
    │
    └── SSH Key personal
             │
             ▼
          GitHub
```
Tu clave personal puede tener acceso a muchos repositorios.

**Ahora imagina que tienes un servidor:**
```bash
Servidor
    │
    └── necesita descargar
        proyecto-servidor
```

No sería buena práctica copiar tu clave personal al servidor.

Porque si el servidor es comprometido, el atacante podría potencialmente obtener acceso a todos los repositorios a los que tenga acceso tu clave personal.

**Con una Deploy Key puedes limitarlo:**

```bash
Tu cuenta
   │
   ├── repo-A
   ├── repo-B
   ├── repo-C
   └── repo-D

Servidor
   │
   └── Deploy Key
           │
           └──────► repo-A solamente
```
Esto es una forma de principio de mínimo privilegio.

### **3. Deploy Key vs SSH Key normal**

Esta diferencia es fundamental.

**SSH Key de tu cuenta**

Se configura en:

GitHub → Settings → SSH and GPG keys

**Conceptualmente:**

```bash
SSH Key
   ↓
TU CUENTA DE GITHUB
   ↓
Repositorios a los que TU CUENTA tiene acceso
```

Es apropiada para: Tu PC → GitHub

**Por ejemplo:**

```bash
git clone git@github.com:usuario/proyecto.git
git push
git pull
```

### **Deploy Key**

Se configura dentro de:

**Repositorio → Settings → Deploy keys**

Conceptualmente:

```bash
Deploy Key
   ↓
UN REPOSITORIO ESPECÍFICO
```

**Por ejemplo:**
```bash
Servidor
   ↓
Deploy Key
   ↓
usuario/mi-servidor
```
**GitHub especifica que una Deploy Key solamente puede estar asociada a un repositorio.**

### **4. El caso más habitual: servidor → GitHub**

Este es probablemente el escenario que te interesa.

Imagina que tienes:
```bash
GitHub
└── Proyecto
```

Y tu servidor tiene:

```bash
/home/server/proyecto/
```

Quieres hacer:
```bash
git pull
```

desde el servidor para actualizar la aplicación.

El flujo sería:
```bash
        GitHub
           ▲
           │
           │ SSH
           │
    Deploy Key
           │
           │
       Servidor
           │
           ▼
       git pull
```
En este escenario normalmente solo necesitas permiso de lectura.

**Por eso es recomendable dejar:**

**☑ Allow write access**

**desactivado.**

Las Deploy Keys son de solo lectura por defecto; GitHub permite habilitar escritura explícitamente.

# **5. Vamos a construir una desde cero**

Supongamos que tienes un servidor Ubuntu.

**Primero:**
```bash
            ssh usuario@IP_DEL_SERVIDOR
```
Una vez dentro:

```bash
mkdir -p ~/.ssh
chmod 700 ~/.ssh
```

Ahora generamos una clave específica para ese repositorio.

**Recomiendo Ed25519:**
```bash
ssh-keygen -t ed25519 -C "deploy-proyecto"
```

Te preguntará:

Enter file in which to save the key:

Puedes utilizar:
```bash
/home/usuario/.ssh/proyecto_deploy
```
**Luego:**

Enter passphrase:

Para un servidor automatizado normalmente se deja sin passphrase, porque el servidor necesita utilizar la clave automáticamente.

**Se generarán:**

```bash
~/.ssh/
├── proyecto_registro_deploy
└── proyecto_registro_deploy.pub
```
**La diferencia es crítica:**
```bash
proyecto_deploy
        ↑
     PRIVADA
        ↑
NO compartir
```
y:
```bash
proyecto_deploy.pub
        ↑
      PÚBLICA
        ↑
Se puede registrar en GitHub
```

### **6. Obtener la clave pública**

En el servidor:

            cat ~/.ssh/proyecto_deploy.pub

Obtendrás algo parecido a:

            ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAI... deploy-proyecto

Copias todo el contenido.

**No copies:**

proyecto_registro_deploy

**Solo:**

proyecto_deploy.pub

### **7. Agregarla a GitHub**

Entra al repositorio.

**Luego:**
```bash
Repository
    ↓
Settings
    ↓
Deploy keys
    ↓
Add deploy key
```
**Pon, por ejemplo:**
```bash
Title:
Servidor producción - proyecto 
En Key:

ssh-ed25519 AAAAC3... deploy-proyecto

Y deja:

☐ Allow write access
```
desmarcado.

**Finalmente:**

Add key

GitHub documenta este procedimiento exactamente así.

### **8. Configurar SSH en el servidor**

Ahora viene una parte importante.

**Edita:**

            nano ~/.ssh/config

Agrega:
```bash
Host github-proyecto
    HostName github.com
    User git
    IdentityFile ~/.ssh/proyecto_deploy
    IdentitiesOnly yes
```
Luego:

            chmod 600 ~/.ssh/config

**Esto le dice a SSH:**

Cuando utilice github-proyecto, usa específicamente esta clave.

### **9. Probar la conexión**

Ejecuta:

            ssh -T git@github-proyecto-registro

GitHub debería reconocer la clave.

Aquí hay un detalle interesante: **una Deploy Key no representa a tu usuario personal**. Está asociada al repositorio.

### **10. Clonar el repositorio**

En lugar de:

            git clone git@github.com:usuario/Proyecto.git

usarías el alias:

            git clone git@github-proyecto:ususario/Proyectogit

SSH verá:
```bash
github-proyecto
        ↓
Host github-proyecto
        ↓
github.com
        ↓
~/.ssh/proyecto_deploy
```

Y GitHub comprobará esa clave contra la Deploy Key registrada.

### **11. Después puedes hacer git pull**

**Una vez clonado:**

            cd Proyecto

**Y:**

            git pull

El servidor podrá descargar las actualizaciones.

Por ejemplo:
```bash
Tu PC
  │
  │ git push
  ▼
GitHub
  │
  │ git pull
  ▼
Servidor
```
Esto es muy común en despliegues sencillos.

### **12. ¿Y si quiero que el servidor haga git push?**

Es posible.

Cuando agregas la Deploy Key puedes activar:

☑ Allow write access

**Pero no lo recomiendo salvo que realmente lo necesites.**

Porque entonces:
```bash
Servidor comprometido
        ↓
Deploy Key robada
        ↓
Atacante
        ↓
Puede escribir en el repositorio
```

GitHub advierte que una Deploy Key con escritura tiene permisos muy amplios sobre ese repositorio.

Para un servidor que solamente necesita actualizarse:
```bash
READ ONLY
   ✓
```
es mucho mejor.

### **13. ¿Qué ocurre si tengo 3 repositorios?**

Aquí aparece una de las principales limitaciones.

**Supongamos:**
```bash
GitHub
├── backend
├── frontend
└── database
```
Y un servidor necesita los tres.

**No puedes hacer:**
```bash
                  ┌── backend
Deploy Key ───────┼── frontend
                  └── database
```
Una Deploy Key está diseñada para un único repositorio.

**La solución sería:**
```bash
Servidor
│
├── backend_deploy
├── frontend_deploy
└── database_deploy
```
Y:
```bash
backend_deploy  → repo backend
frontend_deploy → repo frontend
database_deploy → repo database
```
**GitHub incluso recomienda utilizar claves diferentes y aliases SSH cuando un servidor necesita acceder a múltiples repositorios.**


### **14. Deploy Key y tu servidor casero**

Esto conecta directamente con lo que has estado estudiando sobre montar un servidor casero.

Podrías terminar con algo así:
```bash
                    INTERNET
                        │
                        ▼
                 ┌─────────────┐
                 │   GitHub     │
                 │              │
                 │ proyecto.git │
                 └──────┬───────┘
                        │
                    SSH / Git
                        │
                  Deploy Key
                        │
                        ▼
             ┌────────────────────┐
             │   SERVIDOR CASERO  │
             │                    │
             │ Ubuntu             │
             │ Docker             │
             │                    │
             │ ┌────────────────┐ │
             │ │ Aplicación     │ │
             │ └────────────────┘ │
             └────────────────────┘
```
Y podrías tener un proceso:
```bash
git pull
docker compose up -d --build
```
Entonces:
```bash
Tú
 │
 │ git push
 ▼
GitHub
 │
 │
 ▼
Servidor
 │
 ├── git pull
 │
 ├── docker compose build
 │
 └── docker compose up -d
```
Eso ya se acerca bastante a un deployment automatizado.

### **15. Una cosa muy importante: Deploy Key ≠ SSH para entrar al servidor**

Son dos cosas diferentes.

Puedes tener:
```bash
SSH para administrar el servidor
Tu PC
  │
  │ SSH
  ▼
Servidor
```
Por ejemplo:

            ssh usuario@192.***.*.**

Y simultáneamente:

Deploy Key para GitHub
```bash
Servidor
  │
  │ SSH
  ▼
GitHub
```
Por ejemplo:

        git pull

Son dos relaciones SSH distintas:
```bash
             SSH
Tu PC ─────────────────► Servidor
                           │
                           │ SSH
                           ▼
                        GitHub
```

Esto es algo que conviene que tengas muy claro.

### **16. Seguridad: regla de oro**

En el servidor:

            ~/.ssh/proyecto_deploy

🔴 NUNCA debe subirse a GitHub.

Mientras que:

            ~/.ssh/proyecto_deploy.pub

🟢 sí se registra en GitHub.

Por ejemplo:

# PRIVADA
            cat ~/.ssh/proyecto_deploy

No deberías compartir ese contenido.

La pública:

            cat ~/.ssh/proyecto_deploy.pub

sí puede darse a GitHub.

### **7. ¿Deploy Key, Personal SSH Key o GitHub App?**

Para empezar con servidores, piensa así:

|Método	|Uso|
|:-:|:-:|
|SSH Key personal|	Tu PC → GitHub|
|Deploy Key|	Servidor → 1 repositorio|
|Machine User|	Servidor → varios repositorios|
|GitHub App|	Automatización más avanzada|

GitHub actualmente recomienda considerar GitHub Apps cuando necesitas un control de permisos más fino y una solución más escalable.

sin embargo, Deploy Keys son excelentes para aprender SSH + Git + servidores.

### **Piensa en:**

una Deploy Key como una credencial SSH específica para una máquina y un repositorio:
```bash
                 GITHUB
                    │
             ┌──────┴──────┐
             │             │
        Mi cuenta       Deploy Key
             │             │
        muchos repos   UN repositorio
             │             │
             ▼             ▼
          Mi PC       Mi servidor
```

Y la regla más importante:

### **La clave privada vive en el servidor; la clave pública se registra en GitHub.**

### **Una práctica completa con tu Servidor**

1. Crear deploy_key con ssh-keygen.
2. Configurar ~/.ssh/config.
3. Añadirla a un repositorio privado de GitHub.
4. Probar ssh -T.
5. Clonar el repositorio.
6. Hacer git pull desde el servidor.
7. Ver qué ocurre si intentas acceder a otro repositorio.
8. Finalmente, conectar esto con Docker Compose para desplegar automáticamente tu proyecto.